# Trabalho Prático Final - Parte 2
## Avaliação Final dos Modelos Otimizados

Objetivo: comparar os modelos otimizados no conjunto de teste, usando métricas finais, matriz de confusão, curva ROC e curva Precision-Recall.

### Carregamento de Bibliotecas e Configurações Iniciais

Nesta célula, importamos as bibliotecas necessárias para a análise, como `pandas` para manipulação de dados, `numpy` para operações numéricas, `matplotlib` e `seaborn` para visualização, e módulos do `sklearn` para avaliação de modelos. Também configuramos uma semente aleatória (`RANDOM_SEED`) para reprodutibilidade e criamos os diretórios para salvar resultados e figuras.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib
import warnings

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
)

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

os.makedirs("../results/figures/evaluation", exist_ok=True)
os.makedirs("../results/metrics", exist_ok=True)

### Carregamento e Preparação dos Dados de Teste

Esta célula é responsável por carregar o conjunto de dados de teste (`t2_test.csv`) em um DataFrame do pandas. Ele também separa as características (`X_test`) da variável alvo (`y_test`). A célula inclui um mecanismo para tentar carregar o arquivo tanto de um caminho relativo (para uso local) quanto de um caminho no Google Drive (para uso no Colab, após a montagem do Drive). **Lembre-se de ajustar a variável `base_drive_path` com o caminho correto do seu projeto no Google Drive.**

In [ ]:
target_col = "Overall"

test_df = pd.read_csv("../data/processed/t2_test.csv")

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

print(f"X_test: {X_test.shape}")
print(f"y_test: {y_test.shape}")
print(y_test.value_counts())

### Carregamento dos Modelos Otimizados

Nesta etapa, carregamos os modelos de machine learning previamente otimizados que foram salvos como arquivos `.pkl`. Os modelos são armazenados em um dicionário para facilitar o acesso e a iteração durante a avaliação. Os modelos esperados são 'Gradient Boosting', 'Decision Tree' e 'Random Forest'.

In [ ]:
model_paths = {
    "Gradient Boosting": "../results/models/gradient_boosting_best_model.pkl",
    "Decision Tree": "../results/models/decision_tree_best_model.pkl",
    "Random Forest": "../results/models/random_forest_best_model.pkl"
}

models = {}

for name, path in model_paths.items():
    models[name] = joblib.load(path)
    print(f"Modelo carregado: {name}")

### Avaliação dos Modelos e Geração de Previsões

Esta célula itera sobre cada modelo carregado, faz previsões (`y_pred`) e probabilidades (`y_proba`) no conjunto de teste (`X_test`). Em seguida, calcula métricas de avaliação chave como acurácia, precisão, recall, F1-score e AUC-ROC para cada modelo, armazenando os resultados em um DataFrame. Por fim, salva essas métricas em um arquivo CSV.

In [ ]:
evaluation_results = []

predictions = {}

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    predictions[name] = {
        "y_pred": y_pred,
        "y_proba": y_proba
    }

    result = {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba)
    }

    evaluation_results.append(result)

evaluation_df = pd.DataFrame(evaluation_results)
evaluation_df = evaluation_df.sort_values("f1", ascending=False)

evaluation_df.to_csv("../results/metrics/t2_final_test_metrics.csv", index=False)
evaluation_df

### Relatórios de Classificação Detalhados

Para cada modelo, esta célula imprime um relatório de classificação completo. Este relatório fornece uma visão detalhada da precisão, recall e F1-score para cada classe (não mutagênico e mutagênico), além do suporte (número de instâncias) para cada classe. Isso ajuda a entender o desempenho do modelo em relação a cada tipo de classificação.

In [ ]:
for name in models.keys():
    print("=" * 80)
    print(name)
    print("=" * 80)
    print(classification_report(y_test, predictions[name]["y_pred"]))

### Visualização das Matrizes de Confusão

Nesta célula, geramos e plotamos as matrizes de confusão para cada modelo. A matriz de confusão é uma ferramenta visual que mostra o número de verdadeiros positivos, verdadeiros negativos, falsos positivos e falsos negativos. Isso é crucial para entender os tipos de erros que cada modelo está cometendo e é salva como uma imagem PNG.

In [ ]:
for name in models.keys():
    cm = confusion_matrix(y_test, predictions[name]["y_pred"])

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Não mutagênico", "Mutagênico"],
        yticklabels=["Não mutagênico", "Mutagênico"]
    )

    plt.title(f"Matriz de Confusão - {name}")
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.tight_layout()

    filename = name.lower().replace(" ", "_")
    plt.savefig(f"../results/figures/evaluation/confusion_matrix_{filename}.png", dpi=300)
    plt.show()

### Plotagem das Curvas ROC

Esta célula gera as Curvas Característica de Operação do Receptor (ROC) para cada modelo. As curvas ROC são úteis para visualizar o trade-off entre a taxa de verdadeiros positivos (sensibilidade) e a taxa de falsos positivos (1 - especificidade) em diferentes limiares de classificação. Uma área sob a curva (AUC) maior indica um melhor desempenho geral do modelo. As curvas são salvas como uma imagem PNG.

In [ ]:
plt.figure(figsize=(8, 6))

for name in models.keys():
    RocCurveDisplay.from_predictions(
        y_test,
        predictions[name]["y_proba"],
        name=name,
        ax=plt.gca()
    )

plt.title("Curvas ROC - Modelos Otimizados")
plt.grid(True)
plt.tight_layout()
plt.savefig("../results/figures/evaluation/roc_curves_optimized_models.png", dpi=300)
plt.show()

### Plotagem das Curvas Precision-Recall

Esta célula plota as Curvas Precision-Recall para cada modelo. Essas curvas são particularmente úteis em conjuntos de dados desbalanceados, onde a classe minoritária é de maior interesse. Elas mostram o trade-off entre precisão (proporção de verdadeiros positivos entre todos os positivos previstos) e recall (proporção de verdadeiros positivos entre todos os positivos reais). As curvas são salvas como uma imagem PNG.

In [ ]:
plt.figure(figsize=(8, 6))

for name in models.keys():
    PrecisionRecallDisplay.from_predictions(
        y_test,
        predictions[name]["y_proba"],
        name=name,
        ax=plt.gca()
    )

plt.title("Curvas Precision-Recall - Modelos Otimizados")
plt.grid(True)
plt.tight_layout()
plt.savefig("../results/figures/evaluation/pr_curves_optimized_models.png", dpi=300)
plt.show()

### Comparação Visual das Métricas de Avaliação

Esta célula cria um gráfico de barras para comparar visualmente as métricas de avaliação (acurácia, precisão, recall, F1-score e ROC-AUC) de todos os modelos. Isso permite uma comparação rápida e fácil do desempenho relativo de cada modelo em diferentes aspectos. O gráfico é salvo como uma imagem PNG.

In [ ]:
metrics_to_plot = ["accuracy", "precision", "recall", "f1", "roc_auc"]

evaluation_melted = evaluation_df.melt(
    id_vars="model",
    value_vars=metrics_to_plot,
    var_name="metric",
    value_name="score"
)

plt.figure(figsize=(12, 6))
sns.barplot(
    data=evaluation_melted,
    x="metric",
    y="score",
    hue="model"
)

plt.title("Comparação de métricas no conjunto de teste")
plt.xlabel("Métrica")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.legend(title="Modelo")
plt.tight_layout()
plt.savefig("../results/figures/evaluation/final_metrics_comparison.png", dpi=300)
plt.show()

### Seleção e Salvamento do Melhor Modelo

Com base nos resultados da avaliação, esta célula identifica o modelo com o melhor F1-score no conjunto de teste. O nome do melhor modelo é então impresso e salvo em um arquivo de texto, indicando qual modelo foi escolhido como o final para a tarefa.

In [ ]:
best_model_name = evaluation_df.iloc[0]["model"]
best_model = models[best_model_name]

print(f"Melhor modelo pelo F1-score no teste: {best_model_name}")

with open("../results/metrics/best_model_name.txt", "w", encoding="utf-8") as f:
    f.write(best_model_name)

### Análise de Erros do Melhor Modelo

Esta célula realiza uma análise detalhada dos erros do melhor modelo. Ela cria um novo DataFrame (`error_df`) que inclui as previsões do modelo, as probabilidades e um indicador se a previsão foi correta ou se foi um erro (Falso Negativo ou Falso Positivo). O número de ocorrências de cada tipo de erro é impresso, e o DataFrame de erros é salvo em um CSV para análise posterior.

In [ ]:
best_pred = predictions[best_model_name]["y_pred"]
best_proba = predictions[best_model_name]["y_proba"]

error_df = X_test.copy()
error_df["y_true"] = y_test.values
error_df["y_pred"] = best_pred
error_df["proba_mutagenic"] = best_proba
error_df["correct"] = error_df["y_true"] == error_df["y_pred"]
error_df["error_type"] = np.where(
    error_df["correct"],
    "Acerto",
    np.where(
        (error_df["y_true"] == 1) & (error_df["y_pred"] == 0),
        "Falso Negativo",
        "Falso Positivo"
    )
)

print(error_df["error_type"].value_counts())

error_df.to_csv("../results/metrics/t2_best_model_error_analysis.csv", index=False)

### Distribuição de Probabilidades para Erros

Finalmente, esta célula gera um histograma da distribuição das probabilidades preditas de ser mutagênico para o melhor modelo, diferenciando entre previsões corretas e incorretas. Este gráfico ajuda a entender onde o modelo está mais ou menos confiante e como essas confianças se relacionam com os erros. O gráfico é salvo como uma imagem PNG.

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    data=error_df,
    x="proba_mutagenic",
    hue="correct",
    bins=30,
    kde=True
)

plt.title(f"Distribuição das probabilidades - {best_model_name}")
plt.xlabel("Probabilidade predita de ser mutagênico")
plt.ylabel("Frequência")
plt.tight_layout()
plt.savefig("../results/figures/evaluation/probability_distribution_errors.png", dpi=300)
plt.show()

Nesta etapa, os modelos otimizados foram avaliados no conjunto de teste, mantido isolado durante a etapa de otimização de hiperparâmetros. A comparação foi realizada por meio das métricas Accuracy, Precision, Recall, F1-score e ROC-AUC. O modelo final foi escolhido com base no F1-score no conjunto de teste, por ser uma métrica adequada para problemas com desbalanceamento moderado entre classes.

Além das métricas numéricas, foram analisadas matrizes de confusão, curvas ROC e curvas Precision-Recall. Também foi realizada uma análise agregada dos erros, distinguindo falsos positivos e falsos negativos.